In [1]:
import os
import json
import glob
import pandas as pd
import xlrd
import re
from pathlib import Path
import numpy as np
import xlsxwriter
import openpyxl
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import Alignment, Border, Side, PatternFill

In [2]:
def apply_join(df_ML, df_state, df_samples, df_districts, df_crops):
    #df_ML left join df_state ON  'STATE'
    for feature in ['STATENAME', 'SHORTSTATE', 'HSTATENAME']:
        df_ML[feature] = (df_ML['STATE']
                .map(df_state.set_index('STATE')[feature])
                .fillna(df_ML[feature])
            )
    #df_ML left join df_samples ON  'SAMPLE'
    for feature in ['SAMPLENAME', 'HSAMPLENAME']:
        df_ML[feature] = (df_ML['SAMPLE']
                .map(df_samples.set_index('SAMPLE')[feature])
                .fillna(df_ML[feature])
            )
    #df_ML left join df_districts ON  'DISTRICT'
    for feature in ['DISTRICTNAME', 'HDISTRICTNAME', 'ROCODE', 'RONAME', 'HRONAME', 'SROCODE', 'SRONAME', 'HSRONAME']:
        df_ML[feature] = (df_ML['DISTRICT']
                .map(df_districts.set_index('DISTRICT')[feature])
                .fillna(df_ML[feature])
            )
    #df_ML left join df_crops ON  'CROPCODE'
    for feature in ['CROPNAME', 'HCROPNAME', 'SEASONNAME', 'HSEASONNAME']:
        df_ML[feature] = (df_ML['CROPCODE']
                .map(df_crops.set_index('CROPCODE')[feature])
                .fillna(df_ML[feature])
            )
    return df_ML

In [3]:
def get_first_row(worksheet, row_text_list):
    for text in row_text_list:
        text = text.lower()
        for row in worksheet.iter_rows():
            for cell in row:
                if cell.value and text in str(cell.value).lower():
                    return cell.row
    return -1

In [4]:
def get_first_col(worksheet, col_text_list):
    for text in col_text_list:
        text = text.lower()
        for row in worksheet.iter_rows():
            for cell in row:
                if cell.value and text in str(cell.value).lower():
                    return cell.column
    return -1

In [5]:
def correct_distt_name(incorrect_distt):
    with open("distt_correction.json", "r") as file:
        distt_correction = json.load(file)

    for correct, incorrect_list in distt_correction.items():
        for incorrect in incorrect_list:
            if incorrect.lower() in incorrect_distt.lower():
                correct_distt = correct
                return correct_distt

    return incorrect_distt

In [6]:
def get_irr_cd(crop_raw, crop_type):
    irr_cd = '00'
    if crop_raw is not None and 'paddy' not in crop_raw.lower():
        if crop_type is not None:
            if 'ui' in crop_type.lower():
                irr_cd = '11'
            elif 'I' in crop_type:
                irr_cd = '12'
    return irr_cd

In [7]:
def get_season_cd(crop_raw, crop_type):
    season_cd = None
    kh = '1'
    rb = '4'
    kh_keywords1 = ('kar','kur','sam')
    rb_keywords1 = ('nav','rai')
    
    kh_keywords2 = ('K','sugar','can','maiz','ze','red','dgram')
    rb_keywords2 = ('R','®','hor','rse','segram','hour','green','engram','kgram','black')


    if crop_type is not None and any(k in crop_type.lower() for k in kh_keywords1):
        season_cd = kh
    elif crop_type is not None and any(r in crop_type.lower() for r in rb_keywords1):
        season_cd = rb
    elif 'jowar' in crop_raw.lower():
        temp = crop_raw.lower().replace('jowar','')
        if 'k' in temp:
            season_cd = kh
        elif any(r in temp for r in ('r','®')):
            season_cd = rb
        else:
            season_cd = None
    elif 'K' in crop_raw or any(k in crop_raw.lower() for k in kh_keywords2):
        season_cd = kh
    elif 'R' in crop_raw or any(r in crop_raw.lower() for r in rb_keywords2):
        season_cd = rb
    
    return season_cd

In [8]:
def crop_nm_to_cd(df_crops, season_cd, crop_nm):
    crop_cd = None
    if season_cd is not None:
        df_filtered = df_crops[df_crops['SEASONCODE']==season_cd]
        df_filtered = df_filtered[df_filtered['CROPNAME'].str.contains(crop_nm)]
        crop_cd = str(df_filtered['CROPCD'].iloc[0])
    
    return crop_cd

In [9]:
def get_crop_cd(df_crops, crop_raw, crop_type):
    #get season code using crop_raw and crop_type
    crop_cd = None
    season_cd = get_season_cd(crop_raw, crop_type)
    if season_cd is not None:
        with open("crop_correction.json", "r") as file:
            crop_correction = json.load(file)
    
        crop_nm = crop_raw
        for correct, incorrect_list in crop_correction.items():
            for incorrect in incorrect_list:
                if 'paddy' in crop_raw.lower() and 'paddy' in correct.lower():
                    if incorrect.lower() in crop_type.lower():
                        crop_nm = correct
                elif 'paddy' not in crop_raw.lower() and 'paddy' not in correct.lower():
                    if incorrect.lower() in crop_raw.lower():
                        crop_nm = correct
    
        #get 2-digit crop code from crop name and season
        crop_cd = crop_nm_to_cd(df_crops, season_cd, crop_nm)
    
    return crop_cd, crop_nm, season_cd

In [10]:
def get_crop(df_crops, ws, crop_row, col_idx, crop_type_row):
    crop_raw = ws.cell(row=crop_row, column=col_idx).value
    
    if isinstance(crop_raw, str) and crop_raw is not None:
        crop_raw = crop_raw.strip().replace(' ','')
    else:
        crop_raw = ws.cell(row=crop_row, column=col_idx-1).value
        if isinstance(crop_raw, str) and crop_raw is not None:
            crop_raw = crop_raw.strip().replace(' ','')
        else:
            crop_raw = ws.cell(row=crop_row, column=col_idx-2).value
            if isinstance(crop_raw, str) and crop_raw is not None:
                crop_raw = crop_raw.strip().replace(' ','')
            else:
                crop_raw = None

    if crop_raw is not None:
        crop_raw = re.sub(r'UI', '', crop_raw)
        crop_raw = re.sub(r'IR', '', crop_raw)

        crop_type = ws.cell(row=crop_type_row, column=col_idx).value
        if isinstance(crop_type, str) and crop_type is not None:
            crop_type = crop_type.strip()

        #set irrigation cd [last 2 digits of crop code: _ _ x x]
        irr_cd = get_irr_cd(crop_raw, crop_type)
    
        #set crop cd [1st 2 digits of crop code: x x _ _]
        crop_cd, crop_nm, season_cd = get_crop_cd(df_crops, crop_raw, crop_type)
        
        return crop_cd, crop_nm, season_cd, irr_cd
    return None, None, None, None

In [11]:
def get_SL_to_ML(directory):
    #df to use
    df_state = pd.read_excel("ML_Template.xlsx", sheet_name="State", dtype=str)
    state = df_state['STATE'].iloc[0]
    year = df_state['YEAR'].iloc[0]
    
    df_samples = pd.read_excel("ML_Template.xlsx", sheet_name="Samples", dtype=str)
    df_districts = pd.read_excel("ML_Template.xlsx", sheet_name="Districts", dtype=str)
    df_crops = pd.read_excel("ML_Template.xlsx", sheet_name="Crops", dtype=str)

    #df to create
    df_Vill = pd.DataFrame(columns=['STATE','SEASONCODE','SAMPLE','CROPNAME',
                                    'DISTRICTNAME','BLOCK','VILLAGE'])
    
    df_ML = pd.DataFrame(columns=['YEAR','SEASONCODE','SEASONNAME','HSEASONNAME',
                                  'SAMPLE','SAMPLENAME','HSAMPLENAME','STATE',
                                  'STATENAME','SHORTSTATE','HSTATENAME','ROCODE',
                                  'RONAME','HRONAME','SROCODE','SRONAME','HSRONAME',
                                  'DISTRICT','DISTRICTNAME','HDISTRICTNAME',
                                  'SELORDER','EXPT','CROPCODE','CROPNAME',
                                  'HCROPNAME','STATUS','EXPTID',])
    
    # Get all .xlsx and .xls files in the directory
    excel_files = glob.glob(os.path.join(directory, "*.xlsx"))
    
    for file_path in excel_files:
        file_name = os.path.basename(file_path)
        incorrect_distt = Path(file_name).stem.strip().title()

        district_name = correct_distt_name(incorrect_distt)
        district_code = df_districts[df_districts['DISTRICTNAME']==district_name]['DISTRICT'].iloc[0]
        # Determine file type and use appropriate library
        if file_path.endswith('.xlsx'):
            try:
                wb = load_workbook(file_path, data_only=True)
                for sheet_name in wb.sheetnames:
                    if "cent" in sheet_name.lower() or "sta" in sheet_name.lower():
                        sample = '1' if "cent" in sheet_name.lower() else '2'
                        
                        ws = wb[sheet_name]
                        
                        # setting which cells to scan
                        row_text_list = ['block', 'name of the village', 'name of village']
                        start_row = 0
                        title_row = 0
                        while start_row <= 0:
                            title_row = get_first_row(ws, row_text_list)
                            crop_row = title_row
                            crop_type_row = title_row + 1
                            os_row = title_row + 2
                            start_row = title_row + 3
                            
                        col_text_list = ['Expts. & O.S', 'Expts', 'Expt', 'O.S']
                        start_col = 0
                        while start_col <= 0:
                            start_col = get_first_col(ws, col_text_list)
                            block_col = start_col - 2
                            village_col = start_col - 1
                        
                        end_row = ws.max_row + 1
                        end_col = ws.max_column + 1
    
                        for col_idx in range(start_col, end_col):
                            for row_idx in range(start_row, end_row):
                                cell_val = ws.cell(row=row_idx, column=col_idx).value
                                if cell_val is not None:
                                    if isinstance(cell_val, str):
                                        cell_val = cell_val.strip()
                                        if '-' in cell_val:
                                            req_pattern = str(re.search(r'\d+-\d+', cell_val).group()).strip()
                                            if req_pattern:
                                                expts_n_os = re.findall(r'\d+', req_pattern)
                                                selorder = str(expts_n_os[1])
                                                if len(selorder) == 1:
                                                    selorder = '0' + selorder
                                                
                                                if village_col>0:
                                                    village = ws.cell(row=row_idx, column=village_col).value
                                                    if isinstance(village, str) and village is not None:
                                                        village = village.upper()
                                                        if block_col>0:
                                                            block = ws.cell(row=row_idx, column=block_col).value
                                                            if isinstance(block, str) and block is not None:
                                                                block = block.upper()
                                                                
                                                                # get crop code (2-digit), season code (1-digit) and irrigation code (2-digit)
                                                                crop_cd1, crop_nm, season_cd, irr_cd = get_crop(df_crops, ws, crop_row, col_idx, crop_type_row)
                                                                if crop_cd1 is not None and season_cd is not None:
                                                                    crop_cd = crop_cd1
                                                                    #set crop code for A/B type cotton
                                                                    if crop_cd1 in ('12','55'):
                                                                        if '@' in str(cell_val):
                                                                            crop_cd = '12'
                                                                        else:
                                                                            crop_cd = '55'
                                                                    elif crop_cd1 in ('56','57'):
                                                                        if '@' in str(cell_val):
                                                                            crop_cd = '56'
                                                                        else:
                                                                            crop_cd = '57'
                                                                    crop_code = crop_cd + irr_cd
                                                                    
                                                                    df_village_row = {
                                                                        'STATE': state,
                                                                        'DISTRICTNAME': district_name,
                                                                        'SAMPLE': sample,
                                                                        'SEASONCODE': season_cd,
                                                                        'CROPNAME': crop_nm,
                                                                        'BLOCK': block,
                                                                        'VILLAGE': village
                                                                    }
                                                                    df_Vill = pd.concat([df_Vill, pd.DataFrame([df_village_row])], 
                                                                                        ignore_index=True)
                                                                    
                                                                    df_ML_row = {
                                                                        'YEAR': [year, year],
                                                                        'STATE': [state, state],
                                                                        'DISTRICT': [district_code, district_code],
                                                                        'SAMPLE': [sample, sample],
                                                                        'SEASONCODE': [season_cd, season_cd],
                                                                        'CROPCODE': [crop_code, crop_code],
                                                                        'SELORDER': [selorder, selorder],
                                                                        'EXPT': ['1','2']
                                                                    }
                                                                    
                                                                    df_ML = pd.concat([df_ML, pd.DataFrame(df_ML_row, dtype=str)], ignore_index=True)

            except Exception as e:
                print(f"Error reading {file_name}, {sheet_name}, {row_idx}, {col_idx}, {village}, {crop}")
                raise e

    #joins
    df_ML = apply_join(df_ML, df_state, df_samples, df_districts, df_crops)
    
    return df_Vill, df_ML

In [12]:
if __name__ == "__main__":
    curr_dir = Path.cwd()
    directory = curr_dir / "SL 2.0"
    excel_path = curr_dir / "SL_to_ML.xlsx"
    df_Vill, df_ML = get_SL_to_ML(directory)

    with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
        df_Vill.to_excel(writer, sheet_name="Villages", index=False)
        df_ML.to_excel(writer, sheet_name="ML", index=False)